# Pattern-level simulation of block-wise chord data

Build the data the way it is actually acquired, then let crossnobis recover the geometry:

1. **8 ground-truth chord patterns** shared by the group -- these define the true representational geometry.
2. Each of **10 participants** randomly assigns 4 chords to *trained*, 4 to *untrained*.
3. An optional **training effect** rescales a participant's trained-chord patterns.
4. **Block-wise acquisition**: each run measures every chord once, with fresh measurement noise.

The output is, per participant, a `betas` matrix with `cond_vec` / `part_vec` labels -- ready for
`pcm.est_G_crossval`. Plain code, no functions, so you can step through and tweak each stage.

In [1]:
import numpy as np

rng = np.random.default_rng(0)

n_subj = 10      # participants
n_cond = 8       # chords
n_run  = 10       # runs / blocks (each measures every chord once)
n_vox  = 100     # voxels

signal     = 1.0   # amplitude of the ground-truth patterns
noise      = 1.0   # per-measurement noise sd
subj_var   = 0.0   # >0 adds participant-specific geometry (individual differences)
train_gain = 1.0   # >1 makes trained chords more distinct (training effect); 1.0 = null

## 1. Ground-truth patterns

`U_true` is `(8, n_vox)`: one mean activity pattern per chord, shared by everyone. Drawing the
chords i.i.d. makes them exchangeable, so *any* 4-chord subset has the same expected geometry --
which is what justifies comparing a participant's (randomly chosen) trained vs untrained chords.
The true geometry is `U_true @ U_true.T / n_vox`.

In [2]:
U_true = signal * rng.standard_normal((n_cond, n_vox))
U_true.shape

(8, 100)

## 2. Trained / untrained assignment

Each participant independently picks 4 of the 8 chords as trained (`True`), the rest untrained.

In [3]:
is_trained = np.zeros((n_subj, n_cond), dtype=bool)
for s in range(n_subj):
    trained = rng.choice(n_cond, size=4, replace=False)
    is_trained[s, trained] = True

is_trained.astype(int)

array([[1, 1, 0, 1, 0, 0, 0, 1],
       [0, 0, 0, 0, 1, 1, 1, 1],
       [1, 1, 1, 0, 0, 0, 0, 1],
       [0, 0, 1, 0, 1, 0, 1, 1],
       [1, 1, 1, 0, 0, 0, 1, 0],
       [1, 1, 1, 0, 0, 0, 0, 1],
       [0, 1, 1, 1, 0, 1, 0, 0],
       [1, 0, 0, 1, 0, 1, 0, 1],
       [0, 1, 1, 0, 0, 1, 1, 0],
       [0, 1, 0, 1, 0, 1, 1, 0]])

## 3. Per-participant true patterns (+ training effect)

Start from the shared ground truth, optionally add participant-specific deviations (`subj_var`),
then apply the training effect: multiplying a trained chord's pattern by `train_gain` scales its
pairwise distances by `train_gain**2`, so the trained block's crossnobis distances grow while the
untrained block is untouched. With `train_gain = 1.0` this is the null (no training effect).

In [4]:
U_subj = U_true[None] + subj_var * rng.standard_normal((n_subj, n_cond, n_vox))
for s in range(n_subj):
    U_subj[s, is_trained[s]] *= train_gain

U_subj.shape

(10, 8, 100)

## 4. Block-wise acquisition

For every participant, each run measures all 8 chords once, so a participant has `n_run * n_cond`
rows. `cond_vec` labels the chord, `part_vec` labels the run (the partition for cross-validation).
Fresh Gaussian noise is added to every measurement.

In [5]:
cond_vec = np.tile(np.arange(n_cond), n_run)     # chord id for each row
part_vec = np.repeat(np.arange(n_run), n_cond)   # run id for each row

betas = np.empty((n_subj, n_run * n_cond, n_vox))
for s in range(n_subj):
    betas[s] = U_subj[s, cond_vec] + noise * rng.standard_normal((n_run * n_cond, n_vox))

print('betas   ', betas.shape, '(subj, run*cond, vox)')
print('cond_vec', cond_vec.shape, cond_vec[:n_cond * 2])
print('part_vec', part_vec.shape, part_vec[:n_cond * 2])

betas    (10, 80, 100) (subj, run*cond, vox)
cond_vec (80,) [0 1 2 3 4 5 6 7 0 1 2 3 4 5 6 7]
part_vec (80,) [0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1]
